In [2]:
from pathlib import Path
import pandas as pd
import torch

from voice_project_code import (
    ProjectPaths,
    append_new_allow_speaker,
    build_recording_metadata,
    build_spectrogram_pngs,
    make_loaders,
    predict_wav_file,
    rebuild_after_new_person,
    run_experiment,
    segment_recordings,
    set_seed,
    validate_recording_metadata,
    validate_segment_metadata,
)

SOURCE_DIR = Path(r"VOiCES_devkit/source-16k")
PATHS = ProjectPaths.from_work_dir("voice_project_artifacts")

SEED = 123
ALLOW_SPEAKERS = None
N_ALLOW_SPEAKERS = 5

SEGMENT_SECONDS = 3.0
KEEP_REMAINDER = False
TRIM_SILENCE = False
TOP_DB = 30.0

IMAGE_SIZE = 128
N_FFT = 1024
HOP_LENGTH = 256
N_MELS = 128
FMIN = 20
FMAX = 8000

BATCH_SIZE = 32
EPOCHS = 10
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

set_seed(SEED)
print("DEVICE =", DEVICE)
print("WORK DIR =", PATHS.work_dir.resolve())

DEVICE = cpu
WORK DIR = C:\Users\idani\PycharmProjects\ComputerVision\voice_project_artifacts


In [3]:
recording_meta = build_recording_metadata(
    source_dir=SOURCE_DIR,
    paths=PATHS,
    allow_speakers=ALLOW_SPEAKERS,
    n_allow_speakers=N_ALLOW_SPEAKERS,
    seed=SEED,
)

recording_summary = validate_recording_metadata(recording_meta)

print("recording_meta saved to:", PATHS.recording_meta_path)
print(recording_meta.head())
print()
print(pd.Series(recording_summary))

recording_meta saved to: voice_project_artifacts\metadata_recording_level.csv
  speaker  label split                                         audio_path
0  sp0118      0  test  VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...
1  sp0118      0  test  VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...
2  sp0154      0  test  VOiCES_devkit\source-16k\sp0154\Lab41-SRI-VOiC...
3  sp0154      0  test  VOiCES_devkit\source-16k\sp0154\Lab41-SRI-VOiC...
4  sp0174      0  test  VOiCES_devkit\source-16k\sp0174\Lab41-SRI-VOiC...

n_recordings                                                                  600
split_label_counts              {('test', 0): 118, ('test', 1): 5, ('train', 0...
allow_speakers_per_split                                  {'test': 5, 'train': 5}
not_allow_speakers_per_split              {'test': 59, 'train': 206, 'valid': 30}
dtype: object


In [4]:
segment_meta = segment_recordings(
    paths=PATHS,
    segment_seconds=SEGMENT_SECONDS,
    keep_remainder=KEEP_REMAINDER,
    trim_silence=TRIM_SILENCE,
    top_db=TOP_DB,
    clean_output=True,
)

segment_summary = validate_segment_metadata(segment_meta)

print("segment_meta saved to:", PATHS.segment_meta_path)
print(segment_meta.head())
print()
print(pd.Series(segment_summary))

C:\Users\idani\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


segment_meta saved to: voice_project_artifacts\metadata_segment_level.csv
  speaker  label split                                  source_audio_path  \
0  sp0118      0  test  VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...   
1  sp0118      0  test  VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...   
2  sp0118      0  test  VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...   
3  sp0118      0  test  VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...   
4  sp0118      0  test  VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...   

                                        segment_path  segment_index  \
0  voice_project_artifacts\segmented_audio\test\l...              0   
1  voice_project_artifacts\segmented_audio\test\l...              1   
2  voice_project_artifacts\segmented_audio\test\l...              2   
3  voice_project_artifacts\segmented_audio\test\l...              3   
4  voice_project_artifacts\segmented_audio\test\l...              4   

   start_sample  end_sample  start_s

In [5]:
spectro_meta = build_spectrogram_pngs(
    paths=PATHS,
    image_size=IMAGE_SIZE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    fmin=FMIN,
    fmax=FMAX,
    clean_output=True,
)

print("spectro_meta saved to:", PATHS.spectro_meta_path)
print(spectro_meta.head())
print()
print(spectro_meta.groupby(["split", "label"]).size())

spectro_meta saved to: voice_project_artifacts\metadata_spectrogram_level.csv
  speaker  label split                                  source_audio_path  \
0  sp0118      0  test  VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...   
1  sp0118      0  test  VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...   
2  sp0118      0  test  VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...   
3  sp0118      0  test  VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...   
4  sp0118      0  test  VOiCES_devkit\source-16k\sp0118\Lab41-SRI-VOiC...   

                                        segment_path  \
0  voice_project_artifacts\segmented_audio\test\l...   
1  voice_project_artifacts\segmented_audio\test\l...   
2  voice_project_artifacts\segmented_audio\test\l...   
3  voice_project_artifacts\segmented_audio\test\l...   
4  voice_project_artifacts\segmented_audio\test\l...   

                                          image_path  
0  voice_project_artifacts\spectrograms_png\test\...  
1  voice_pro

In [6]:
train_loader, valid_loader, test_loader, loader_stats = make_loaders(
    paths=PATHS,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    normalize=True,
    augment=False,
)

print(pd.Series(loader_stats))

train_mean                    0.338138
train_std                     0.214267
class_counts_train    {0: 1996, 1: 25}
n_train                           2021
n_valid                            291
n_test                             603
dtype: object


In [7]:
history_baseline, results_baseline = run_experiment(
    paths=PATHS,
    experiment_name="baseline_smallcnn_adam",
    model_name="smallcnn",
    dropout=0.30,
    use_batchnorm=True,
    optimizer_name="adam",
    lr=1e-3,
    weight_decay=1e-4,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    normalize=True,
    augment=False,
    device=DEVICE,
)

display(history_baseline.tail())
display(results_baseline)

,epoch,train_loss,train_acc,train_far,train_frr,valid_acc,valid_far,valid_frr,valid_score_far_plus_frr
5,6,0.444194,0.935675,0.058116,0.56,0.910653,0.089347,0.0,0.089347
6,7,0.432448,0.933201,0.061122,0.52,0.896907,0.103093,0.0,0.103093
7,8,0.455202,0.847105,0.151804,0.24,0.731959,0.268041,0.0,0.268041
8,9,0.385035,0.982682,0.007014,0.84,0.979381,0.020619,0.0,0.020619
9,10,0.361238,0.902029,0.095691,0.28,0.828179,0.171821,0.0,0.171821


,experiment_name,model_name,optimizer_name,dropout,use_batchnorm,normalize,augment,epochs,batch_size,lr,...,valid_far,valid_frr,test_acc,test_far,test_frr,train_mean,train_std,n_train,n_valid,n_test
0,baseline_smallcnn_adam,smallcnn,adam,0.3,True,True,False,10,32,0.001,...,0.010309,0.0,0.958541,0.0,1.0,0.338138,0.214267,2021,291,603


In [8]:
EXPERIMENT_CONFIGS = [
    {
        "experiment_name": "exp_smallcnn_adam_bn_dropout",
        "model_name": "smallcnn",
        "dropout": 0.30,
        "use_batchnorm": True,
        "optimizer_name": "adam",
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMAGE_SIZE,
        "normalize": True,
        "augment": False,
        "device": DEVICE,
    },
    {
        "experiment_name": "exp_smallcnn_sgd_bn_dropout",
        "model_name": "smallcnn",
        "dropout": 0.30,
        "use_batchnorm": True,
        "optimizer_name": "sgd",
        "lr": 1e-2,
        "weight_decay": 1e-4,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMAGE_SIZE,
        "normalize": True,
        "augment": False,
        "device": DEVICE,
    },
    {
        "experiment_name": "exp_smallcnn_adam_no_bn",
        "model_name": "smallcnn",
        "dropout": 0.30,
        "use_batchnorm": False,
        "optimizer_name": "adam",
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMAGE_SIZE,
        "normalize": True,
        "augment": False,
        "device": DEVICE,
    },
    {
        "experiment_name": "exp_smallcnn_adam_no_dropout",
        "model_name": "smallcnn",
        "dropout": 0.00,
        "use_batchnorm": True,
        "optimizer_name": "adam",
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMAGE_SIZE,
        "normalize": True,
        "augment": False,
        "device": DEVICE,
    },
    {
        "experiment_name": "exp_resnet18_scratch",
        "model_name": "resnet18_scratch",
        "dropout": 0.30,
        "use_batchnorm": True,
        "optimizer_name": "adam",
        "lr": 1e-3,
        "weight_decay": 1e-4,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "image_size": IMAGE_SIZE,
        "normalize": True,
        "augment": False,
        "device": DEVICE,
    },
    # Optional:
    # {
    #     "experiment_name": "exp_resnet18_transfer",
    #     "model_name": "resnet18_transfer",
    #     "dropout": 0.30,
    #     "use_batchnorm": True,
    #     "optimizer_name": "adam",
    #     "lr": 1e-4,
    #     "weight_decay": 1e-4,
    #     "epochs": EPOCHS,
    #     "batch_size": BATCH_SIZE,
    #     "image_size": IMAGE_SIZE,
    #     "normalize": True,
    #     "augment": False,
    #     "device": DEVICE,
    # },
]

In [9]:
all_results = []

for cfg in EXPERIMENT_CONFIGS:
    _, results_df = run_experiment(paths=PATHS, **cfg)
    all_results.append(results_df)

comparison_df = pd.concat(all_results, ignore_index=True)
comparison_df = comparison_df.sort_values(["valid_far", "valid_frr", "test_far", "test_frr"])

comparison_df

,experiment_name,model_name,optimizer_name,dropout,use_batchnorm,normalize,augment,epochs,batch_size,lr,...,valid_far,valid_frr,test_acc,test_far,test_frr,train_mean,train_std,n_train,n_valid,n_test
0,exp_smallcnn_adam_bn_dropout,smallcnn,adam,0.3,True,True,False,10,32,0.001,...,0.000000,0.0,0.958541,0.00000,1.00,0.338138,0.214267,2021,291,603
1,exp_smallcnn_sgd_bn_dropout,smallcnn,sgd,0.3,True,True,False,10,32,0.010,...,0.000000,0.0,0.958541,0.00000,1.00,0.338138,0.214267,2021,291,603
2,exp_smallcnn_adam_no_bn,smallcnn,adam,0.3,False,True,False,10,32,0.001,...,0.000000,0.0,0.958541,0.00000,1.00,0.338138,0.214267,2021,291,603
4,exp_resnet18_scratch,resnet18_scratch,adam,0.3,True,True,False,10,32,0.001,...,0.000000,0.0,0.958541,0.00346,0.92,0.338138,0.214267,2021,291,603
3,exp_smallcnn_adam_no_dropout,smallcnn,adam,0.0,True,True,False,10,32,0.001,...,0.017182,0.0,0.961857,0.00173,0.88,0.338138,0.214267,2021,291,603


In [ ]:
# Example:
# NEW_PERSON_DIR = Path(r"new_allow_speakers/sp9999")
# updated_recording_meta = append_new_allow_speaker(
#     paths=PATHS,
#     new_person_dir=NEW_PERSON_DIR,
#     seed=SEED,
# )
# updated_segment_meta, updated_spectro_meta = rebuild_after_new_person(
#     paths=PATHS,
#     segment_seconds=SEGMENT_SECONDS,
#     keep_remainder=KEEP_REMAINDER,
#     trim_silence=TRIM_SILENCE,
#     top_db=TOP_DB,
#     image_size=IMAGE_SIZE,
#     n_fft=N_FFT,
#     hop_length=HOP_LENGTH,
#     n_mels=N_MELS,
#     fmin=FMIN,
#     fmax=FMAX,
# )
# 
# history_updated, results_updated = run_experiment(
#     paths=PATHS,
#     experiment_name="baseline_smallcnn_after_new_person",
#     model_name="smallcnn",
#     dropout=0.30,
#     use_batchnorm=True,
#     optimizer_name="adam",
#     lr=5e-4,
#     weight_decay=1e-4,
#     epochs=5,
#     batch_size=BATCH_SIZE,
#     image_size=IMAGE_SIZE,
#     normalize=True,
#     augment=False,
#     device=DEVICE,
# )
# 
# display(results_updated)

In [ ]:
# Example:
# BEST_CHECKPOINT = PATHS.model_dir / "baseline_smallcnn_adam.pt"
# wav_result = predict_wav_file(
#     wav_path=Path(r"some_test_audio.wav"),
#     checkpoint_path=BEST_CHECKPOINT,
#     model_name="smallcnn",
#     train_mean=loader_stats["train_mean"],
#     train_std=loader_stats["train_std"],
#     segment_seconds=SEGMENT_SECONDS,
#     image_size=IMAGE_SIZE,
#     device=DEVICE,
#     dropout=0.30,
#     use_batchnorm=True,
# )
# 
# wav_result